# Paso 9b: exportar el modelo sin el conversor de TensorFlow.js

La conversión con `tensorflowjs` falla con frecuencia en Colab por incompatibilidad de versiones. Este cuaderno la evita: en lugar de convertir el modelo, **exporta solo sus pesos** en un archivo JSON. La página `medir_navegador_pesos.html` reconstruye la misma arquitectura en JavaScript y le asigna esos pesos.

El resultado es idéntico y la página verifica que las predicciones coincidan con las de Python.

Descarga **dos archivos**: `pesos.json` y `latidos.json`.

**Antes de ejecutar:** activa la **T4 GPU**. Tarda unos 10 minutos.

## 1. Instalar y descargar datos

In [ ]:
!pip install wfdb -q
import wfdb, time, json, os
import numpy as np
from scipy.signal import lfilter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

wfdb.dl_database('mitdb', dl_dir='mitdb')
print('Descarga lista')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.6 MB/s eta 0:00:00
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list f

## 2. Preparar los latidos

In [ ]:
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
AAMI = {'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S','V':'V','E':'V','F':'F'}
CODIGOS_LATIDO = set('NLRejAaJSVEF/fQ')
CLASES = ['N','S','V','F']
FS, ANTES, DESPUES = 360, 90, 162

def filtro_pasa_altas(x, fs=FS, fc=1.0):
    alpha = np.exp(-2*np.pi*fc/fs)
    return x - lfilter([1-alpha], [1, -alpha], x)

def extraer_latidos(num):
    ruta = f'mitdb/{num}'
    rec = wfdb.rdrecord(ruta); ann = wfdb.rdann(ruta, 'atr')
    canal = rec.sig_name.index('MLII') if 'MLII' in rec.sig_name else 0
    x = filtro_pasa_altas(rec.p_signal[:, canal]); x = (x - x.mean())/x.std()
    pos = np.array([p for p, s in zip(ann.sample, ann.symbol) if s in CODIGOS_LATIDO])
    sim = [s for s in ann.symbol if s in CODIGOS_LATIDO]
    rr = np.diff(pos)/FS; rr_medio = rr.mean()
    V, R, Y = [], [], []
    for i in range(1, len(pos)-1):
        if sim[i] not in AAMI: continue
        r = pos[i]
        if r-ANTES < 0 or r+DESPUES > len(x): continue
        V.append(x[r-ANTES : r+DESPUES])
        R.append([rr[i-1], rr[i], rr[max(0,i-10):i].mean(), rr[i-1]/rr_medio, rr[i]/rr_medio])
        Y.append(AAMI[sim[i]])
    return V, R, Y

def construir(lista):
    V, R, Y = [], [], []
    for n in lista:
        v, r, y = extraer_latidos(n); V += v; R += r; Y += y
    return np.array(V, dtype='float32'), np.array(R, dtype='float32'), np.array(Y)

V_ent, R_ent, y_ent = construir(DS1)
V_pru, R_pru, y_pru = construir(DS2)
a_numero = {c:i for i,c in enumerate(CLASES)}
y_ent_num = np.array([a_numero[c] for c in y_ent])
y_pru_num = np.array([a_numero[c] for c in y_pru])
V_tr, V_val, R_tr, R_val, y_tr, y_val = train_test_split(
    V_ent, R_ent, y_ent_num, test_size=0.1, stratify=y_ent_num, random_state=42)
print('Listo. Prueba:', len(y_pru_num), 'latidos')

Listo. Prueba: 49661 latidos


## 3. Entrenar la red convolucional

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(0)

ef = keras.Input(shape=(252, 1))
x = layers.Conv1D(32, 7, activation='relu', padding='same')(ef)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(64, 3, activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling1D()(x)
er = keras.Input(shape=(5,))
z = layers.concatenate([x, er])
z = layers.Dense(64, activation='relu')(z)
z = layers.Dropout(0.3)(z)
sal = layers.Dense(4, activation='softmax')(z)
modelo = keras.Model([ef, er], sal)
modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])

pesos_clase = {i: w for i, w in enumerate(
    compute_class_weight('balanced', classes=np.arange(4), y=y_tr))}
modelo.fit([V_tr[..., None], R_tr], y_tr,
           validation_data=([V_val[..., None], R_val], y_val),
           epochs=40, batch_size=256, class_weight=pesos_clase,
           callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
                      keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4)],
           verbose=0)
pred = modelo.predict([V_pru[..., None], R_pru], verbose=0).argmax(1)
print('Parametros:', modelo.count_params())
print('F1 macro en DS2:', round(f1_score(y_pru_num, pred, labels=range(4), average='macro', zero_division=0), 3))

Parametros: 27652
F1 macro en DS2: 0.589


## 4. Latidos de prueba y referencia en Python

In [ ]:
rng = np.random.default_rng(0)
idx = []
for i in range(4):
    disponibles = np.where(y_pru_num == i)[0]
    idx.extend(rng.choice(disponibles, min(125, len(disponibles)), replace=False))
idx = np.array(sorted(idx))

prob_ref = modelo.predict([V_pru[idx][..., None], R_pru[idx]], verbose=0)
exactitud_py = accuracy_score(y_pru_num[idx], prob_ref.argmax(1))
print('Latidos seleccionados:', len(idx))
print('Exactitud de referencia (Python):', round(exactitud_py, 4))

tiempos = []
for k in idx[:200]:
    t0 = time.perf_counter()
    modelo.predict([V_pru[k:k+1][..., None], R_pru[k:k+1]], verbose=0)
    tiempos.append((time.perf_counter()-t0)*1000)
print('Python, latido por latido: mediana', round(float(np.median(tiempos)), 3), 'ms')

Latidos seleccionados: 500
Exactitud de referencia (Python): 0.654
Python, latido por latido: mediana 76.31 ms


## 5. Exportar pesos y latidos

Fíjate en la lista de formas que imprime: deben ser diez tensores, en este orden exacto.

```
[7, 1, 32]  [32]  [5, 32, 64]  [64]  [3, 64, 64]  [64]  [69, 64]  [64]  [64, 4]  [4]
```

Si no coincide, avísame antes de seguir.

In [ ]:
os.makedirs('medicion', exist_ok=True)

# --- Pesos del modelo ---
# Se guardan en el orden exacto en que los devuelve Keras, que es el mismo
# en que la pagina los va a asignar despues de reconstruir la arquitectura.
pesos = [{'forma': list(w.shape), 'valores': [round(float(v), 7) for v in w.flatten()]}
         for w in modelo.get_weights()]
total = sum(len(p['valores']) for p in pesos)
print('Tensores de pesos:', len(pesos), '| valores totales:', total)
for i, p in enumerate(pesos):
    print(f'  {i}: {p["forma"]}')

with open('medicion/pesos.json', 'w') as f:
    json.dump({'parametros': int(total), 'pesos': pesos}, f)

# --- Latidos de prueba ---
datos = {
    'clases': CLASES,
    'exactitud_referencia_python': float(round(exactitud_py, 6)),
    'latidos': [{'forma': [round(float(v), 4) for v in V_pru[k]],
                 'rr':    [round(float(v), 6) for v in R_pru[k]],
                 'clase': int(y_pru_num[k])} for k in idx]
}
with open('medicion/latidos.json', 'w') as f:
    json.dump(datos, f)

for a in ['pesos.json', 'latidos.json']:
    print(a, round(os.path.getsize('medicion/'+a)/1e6, 2), 'MB')

Tensores de pesos: 10 | valores totales: 27652
  0: [7, 1, 32]
  1: [32]
  2: [5, 32, 64]
  3: [64]
  4: [3, 64, 64]
  5: [64]
  6: [69, 64]
  7: [64]
  8: [64, 4]
  9: [4]
pesos.json 0.32 MB
latidos.json 1.1 MB


## 6. Descargar

In [ ]:
from google.colab import files
files.download('medicion/pesos.json')
files.download('medicion/latidos.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>